In [ ]:
"""
Load the RetailRocket raw dataset files for initial inspection.

This script reads the interaction log, item property files, and
category hierarchy file into pandas DataFrames. The two item
property files are combined into a single table because the
dataset stores item metadata across separate CSV files.

Basic shape information is printed to confirm successful loading,
and the first rows of the events table are displayed for a quick
structural check.
"""

# -----------------------------
# 1. Imports
# -----------------------------
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from pathlib import Path
import pickle

# -----------------------------
# 2. Load raw data
# -----------------------------
# Read the user interaction event log.
events = pd.read_csv("events.csv")

# Read both item property files and combine them into one table.
ip1 = pd.read_csv("item_properties_part1.csv")
ip2 = pd.read_csv("item_properties_part2.csv")  # Remove if not present.
item_props = pd.concat([ip1, ip2], ignore_index=True)

# Read the category hierarchy data.
category_tree = pd.read_csv("category_tree.csv")

# -----------------------------
# 3. Basic inspection
# -----------------------------
# Print dataset dimensions to verify that each file loaded correctly.
print("Events shape:", events.shape)
print("Item properties shape:", item_props.shape)
print("Category tree shape:", category_tree.shape)

# Preview the first rows of the events table.
events.head()

Events shape: (2756101, 5)
Item properties shape: (20275902, 4)
Category tree shape: (1669, 2)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [ ]:
# ------------------------------------------------------------
# 2. CLEAN EVENTS AND APPLY IMPLICIT INTERACTION WEIGHTS
# ------------------------------------------------------------

# Retain only interaction types used in the recommendation pipeline.
# Any unexpected or irrelevant event labels are excluded at this stage.
valid_events = ["view", "addtocart", "transaction"]
events = events[events["event"].isin(valid_events)].copy()

# Assign implicit feedback weights to each interaction type.
# Higher weights are given to stronger behavioural signals.
event_weights = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 5.0,
}

# Map each event label to its corresponding implicit weight.
events["weight"] = events["event"].map(event_weights)

# Report the dataset size after filtering to confirm the retained volume.
print("Events after filtering:", events.shape)

Events after filtering: (2756101, 6)


In [ ]:
# ------------------------------------------------------------
# 3. BUILD USER AND ITEM INDICES
# ------------------------------------------------------------

# Extract the distinct item identifiers present in the filtered
# interaction data. The item index is built only from observed events
# so that matrix construction remains aligned with the interaction log.
item_ids = events["itemid"].unique()

# Create forward and reverse lookup dictionaries for item identifiers.
# These mappings convert raw item IDs to contiguous integer indices
# suitable for sparse matrix construction and later decoding.
item_id_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}
idx_to_item_id = {idx: item_id for item_id, idx in item_id_to_idx.items()}
n_items = len(item_ids)

# Extract the distinct user identifiers present in the filtered
# interaction data. These values form the row index of the user-item matrix.
user_ids = events["visitorid"].unique()

# Create forward and reverse lookup dictionaries for user identifiers.
# These mappings allow raw user IDs to be converted into integer indices
# for model training, evaluation, and recommendation lookup.
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
idx_to_user_id = {idx: user_id for user_id, idx in user_id_to_idx.items()}
n_users = len(user_ids)

# Print the number of indexed users and items as a basic validation check.
# The item count is expected to match the filtered interaction-derived total.
print("Users:", n_users)
print("Items:", n_items)  # Expected item count: 235,061

Users: 1407580
Items: 235061


In [ ]:
# ------------------------------------------------------------
# 4. MAP EVENT RECORDS TO MATRIX INDICES
# ------------------------------------------------------------

# Map each visitor identifier to its corresponding internal user index.
events["user_idx"] = events["visitorid"].map(user_id_to_idx)

# Map each item identifier to its corresponding internal item index.
events["item_idx"] = events["itemid"].map(item_id_to_idx)

# Display the first rows to inspect the newly created index columns.
events.head()

,timestamp,visitorid,event,itemid,transactionid,weight,user_idx,item_idx
0,1433221332117,257597,view,355908,NaN,1.0,0,0
1,1433224214164,992329,view,248676,NaN,1.0,1,1
2,1433221999827,111016,view,318965,NaN,1.0,2,2
3,1433221955914,483717,view,253185,NaN,1.0,3,3
4,1433221337106,951259,view,367447,NaN,1.0,4,4


In [ ]:
# ------------------------------------------------------------
# 5. BUILD USER-ITEM INTERACTION MATRIX
# ------------------------------------------------------------

# Aggregate repeated interactions between the same user and item pair.
# Weights are summed so that multiple behavioural signals contribute to
# a single implicit feedback value for each user-item combination.
interaction_df = (
    events.groupby(["user_idx", "item_idx"])["weight"]
    .sum()
    .reset_index()
)

# Construct a sparse user-item interaction matrix from the aggregated data.
# Rows represent users, columns represent items, and each stored value
# represents the total implicit interaction weight for that user-item pair.
interactions = csr_matrix(
    (
        interaction_df["weight"],
        (interaction_df["user_idx"], interaction_df["item_idx"]),
    ),
    shape=(n_users, n_items),
)

# Print the matrix dimensions and the number of non-zero entries
# to confirm successful construction of the interaction matrix.
print("Interaction matrix:", interactions.shape)
print("Non-zero entries:", interactions.nnz)

Interaction matrix: (1407580, 235061)
Non-zero entries: 2145179


In [ ]:
# ------------------------------------------------------------
# 6. FILTER ITEM PROPERTIES TO ITEMS PRESENT IN EVENTS
# ------------------------------------------------------------

# Retain only item property records for items that appear in the
# interaction data. This ensures that subsequent item metadata
# processing remains aligned with the indexed event set.
valid_item_ids = set(item_id_to_idx.keys())

item_props_filtered = item_props[item_props["itemid"].isin(valid_item_ids)].copy()
print("Filtered item_properties:", item_props_filtered.shape)

# Sort item property records by item, property, and timestamp so that
# the most recent snapshot for each item-property pair appears last.
item_props_filtered = item_props_filtered.sort_values(
    ["itemid", "property", "timestamp"]
)

# Keep only the latest available snapshot for each item-property pair.
# This removes older versions of the same property and preserves the
# most recent metadata state for each item.
item_props_latest = (
    item_props_filtered.groupby(["itemid", "property"]).tail(1)
)

# Report the size of the deduplicated latest-snapshot table.
print("Latest snapshots:", item_props_latest.shape)

Filtered item_properties: (10180153, 4)
Latest snapshots: (5369446, 4)


In [ ]:
# ------------------------------------------------------------
# 7. BUILD TEXT FEATURES FOR ITEMS
# ------------------------------------------------------------

# Combine the retained property values for each item into a single
# text representation. This creates one aggregated text field per item,
# which can later be used for content-based feature extraction.
item_text_df = (
    item_props_latest
    .groupby("itemid")["value"]
    .apply(lambda vals: " ".join(map(str, vals)))
    .reset_index()
    .rename(columns={"value": "raw_text"})
)

# Print the number of item-level text records created.
print("Item text rows:", item_text_df.shape)

Item text rows: (185246, 2)


In [ ]:
# ------------------------------------------------------------
# 8. ALIGN TEXT FEATURES WITH ITEM INDICES
# ------------------------------------------------------------

# Map each item identifier in the text feature table to its internal
# item index so that text features align with the indexed item space.
item_text_df["item_idx"] = item_text_df["itemid"].map(item_id_to_idx)

# Remove any rows that could not be matched to a valid indexed item.
# This preserves consistency between the text feature table and the
# interaction-derived item index.
item_text_df = item_text_df.dropna(subset=["item_idx"])

# Convert mapped item indices to integer type after removing missing values.
item_text_df["item_idx"] = item_text_df["item_idx"].astype(int)

# Build the final item reference table using the full indexed item range.
# A left join is used so that every indexed item is retained, even when
# no text features are available for that item.
items = pd.DataFrame(
    {
        "item_idx": np.arange(n_items),
        "itemid": [idx_to_item_id[i] for i in range(n_items)],
    }
).merge(item_text_df, on="item_idx", how="left")

# Replace missing text values with empty strings so that all items have
# a valid text field for later feature extraction.
items["raw_text"] = items["raw_text"].fillna("")

# Report the final item table size and preview the first rows as a
# validation check after alignment.
print("Final items table:", items.shape)  # Expected shape: (235061, 4)
items.head()

Final items table: (235061, 4)


,item_idx,itemid_x,itemid_y,raw_text
0,0,355908,355908.0,726612 n1020.000 424566 679677 519769 264157 2...
1,1,248676,248676.0,961511 679677 519769 769062 857982 540717 1407...
2,2,318965,NaN,
3,3,253185,253185.0,769062 469355 679677 769062 519769 769062 n120...
4,4,367447,367447.0,119932 754228 801383 471403 801383 693640 9860...


In [ ]:
# ------------------------------------------------------------
# 9. SAVE ARTIFACTS FOR SUBSEQUENT NOTEBOOKS
# ------------------------------------------------------------

from scipy import sparse

# Create the artifacts directory if it does not already exist.
# This directory is used to store processed outputs for reuse
# in later stages of the recommendation pipeline.
Path("artifacts").mkdir(exist_ok=True)

# Save the sparse user-item interaction matrix in NPZ format.
# This preserves the matrix efficiently for later loading without
# reconstructing it from the raw event data.
sparse.save_npz("input/artifacts/interactions.npz", interactions)

# Save the aligned item table as a CSV file so that indexed items
# and their associated text features can be reused in later notebooks.
items.to_csv("input/artifacts/items_aligned.csv", index=False)

# Save the user identifier to index mapping for consistent reuse
# across training, inference, and evaluation steps.
with open("input/artifacts/user_id_to_idx.pkl", "wb") as f:
    pickle.dump(user_id_to_idx, f)

# Save the item identifier to index mapping so that raw item IDs
# can be converted back to the indexed representation when needed.
with open("input/artifacts/item_id_to_idx.pkl", "wb") as f:
    pickle.dump(item_id_to_idx, f)

# Confirm that all required artifacts were written successfully.
print("Artifacts saved successfully.")

Artifacts saved successfully.
